# Creating parameters — every type

Parameters are named values inside a part that features can reference
by name. Alibre exposes four parameter types:

| `ADParameterType` | Meaning | Example |
|---|---|---|
| `AD_DISTANCE` | linear length (cm) | wall thickness, depth |
| `AD_ANGLE` | angle (radians) | sweep angle |
| `AD_COUNT` | integer multiplicity | pattern count |
| `AD_SCALE` | dimensionless ratio | scale factor |

**Prereq:** open a fresh empty part in Alibre.

## Setup

In [ ]:
from alibrex import CurrentPart, ADParameterType

part = CurrentPart()
params = part.Parameters
params.Count

## Pattern: transaction wrap

Parameter mutations must run inside a transaction so the part
regenerates consistently. Each cell below opens/closes its own
transaction. You can batch many writes inside one transaction if
you prefer.

## Create a `DISTANCE` parameter

In [ ]:
thickness = params.NewParameter("Thickness", ADParameterType.AD_DISTANCE)
params.OpenParameterTransaction()
thickness.Value = 0.5                # cm
params.CloseParameterTransaction()
thickness.Name, thickness.Value, thickness.Units

## Create an `ANGLE` parameter

In [ ]:
import math

sweep = params.NewParameter("Sweep", ADParameterType.AD_ANGLE)
params.OpenParameterTransaction()
sweep.Value = math.radians(45.0)     # radians!
params.CloseParameterTransaction()
sweep.Name, sweep.Value, sweep.Units

## Create a `COUNT` parameter

In [ ]:
holes = params.NewParameter("HoleCount", ADParameterType.AD_COUNT)
params.OpenParameterTransaction()
holes.Value = 6
params.CloseParameterTransaction()
holes.Name, holes.Value, holes.Units

## Create a `SCALE` parameter

In [ ]:
scale = params.NewParameter("ScaleFactor", ADParameterType.AD_SCALE)
params.OpenParameterTransaction()
scale.Value = 1.25
params.CloseParameterTransaction()
scale.Name, scale.Value, scale.Units

## Drive one parameter by an equation referencing others

Equations are plain strings that can reference any parameter by name.

In [ ]:
total = params.NewParameter("TotalThickness", ADParameterType.AD_DISTANCE)
params.OpenParameterTransaction()
total.Equation = "Thickness * HoleCount"
params.CloseParameterTransaction()
part.RegenerateAll()
total.Value, total.Equation

## Push a base value — equation-driven params follow

In [ ]:
params.OpenParameterTransaction()
thickness.Value = 1.0
params.CloseParameterTransaction()
part.RegenerateAll()
thickness.Value, total.Value

## Comment field — annotate a parameter

In [ ]:
thickness.Comment = "wall thickness; drives downstream features"
thickness.Comment

## Batch many writes inside one transaction

In [ ]:
params.OpenParameterTransaction()
thickness.Value = 0.75
holes.Value = 8
scale.Value = 1.5
params.CloseParameterTransaction()
part.RegenerateAll()
[(p.Name, p.Value) for p in (thickness, holes, scale, total)]

## Cancel a transaction — abandon changes mid-flight

In [ ]:
params.OpenParameterTransaction()
thickness.Value = 999.0              # not committed
params.CancelParameterTransaction()
thickness.Value                      # back to its pre-transaction value

## Final dump — every parameter on the part

In [ ]:
for i in range(params.Count):
    p = params.Item(i)
    eq = f"  =  {p.Equation}" if p.Equation else ""
    print(f"  {p.Name:20s} = {p.Value:10.4f}  [{p.Units}]{eq}")